# Ridge Regression Experiment

This notebook explores Ridge regression for the Runge function, following part (b)
of the project. We repeat the polynomial-degree analysis from `ols_experiment.ipynb`
(part (a)) but now sweep the penalty strength $\lambda$ as well, and connect what we
see to the shrinkage of the singular-value modes of the design matrix (Section 3.8 of
the lecture notes).

First we import everything we need: `Ridge` for the model under study, `OLS` for
the part (a) comparison/sanity check, `StandardScaler`/`train_test_split` from
scikit-learn for preprocessing, and the shared `fys_stk4155_p1` helpers for the
data, the polynomial design matrix, and the MSE/R2 metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error, r2_score
from fys_stk4155_p1.regression.ordinary_least_squares import OLS
from fys_stk4155_p1.regression.ridge import Ridge

We can then generate and visualize the data (same Runge data as part a)).

In [ ]:
# Generate data
x, y = generate_runge_data(n=100, noise_std=0.1, seed=42)

# Dense grid (not the noisy samples) for plotting the smooth ground-truth curve.
x_plot = np.linspace(-1, 1, 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge's function")
ax.scatter(x, y, s=15, alpha=0.6, label=r"data, $\sigma = 0.1$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Runge function: noisy samples vs. ground truth")
ax.legend()
fig.tight_layout()
plt.show()

## Fit Ridge across polynomial degree and $\lambda$

We reuse the exact scaling/splitting scheme from part a): the intercept column of
ones is kept out of `StandardScaler` (fit on the training split only) and out of
the design matrix slicing per degree. The only new ingredient is the penalty
$\lambda$, passed to `Ridge(lam=lam, fit_intercept_column=True)` so the intercept
term itself is never shrunk.

`fit_polynomial_ridge_by_degree` mirrors `fit_polynomial_ols_by_degree` from
`ols_experiment.ipynb`, at a single fixed $\lambda$; we then sweep it over a grid
of $\lambda$ values below.

In [ ]:
def fit_polynomial_ridge_by_degree(
    x: np.ndarray,
    y: np.ndarray,
    degrees: range,
    lam: float,
    test_size: float = 0.2,
    seed: int = 42,
) -> dict:
    """Fit Ridge on polynomial features of x, for each degree in `degrees`, at fixed lam.

    Same design-matrix/scaling/split scheme as `fit_polynomial_ols_by_degree` in
    ols_experiment.ipynb: the intercept column is left unscaled and unpenalized
    (`fit_intercept_column=True`), and the scaler is fit on the training split only.

    Args:
        x: x-coordinates, shape (n,).
        y: targets, shape (n,).
        degrees: polynomial degrees to fit, e.g. range(1, 16).
        lam: Ridge penalty strength (lam=0 recovers OLS).
        test_size: fraction of samples held out for testing.
        seed: seed for the train/test split, for reproducibility.

    Returns:
        Dict with keys "degrees", "weights" (list of theta arrays, one per degree;
        theta[0] is the intercept), "mse_train", "mse_test", "r2_train", "r2_test"
        (arrays aligned with "degrees").
    """
    degrees_arr = np.array(list(degrees))
    max_degree = int(degrees_arr.max())

    X_full = univariate_polynomial_design_matrix(x=x, degree=max_degree)
    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_full, y, test_size=test_size, random_state=seed
    )

    weights = []
    mse_train, mse_test = [], []
    r2_train, r2_test = [], []

    for degree in degrees_arr:
        # Column 0 is the intercept (all ones); columns 1..degree are x^1..x^degree.
        X_train = X_train_full[:, : degree + 1]
        X_test = X_test_full[:, : degree + 1]

        scaler = StandardScaler()
        X_train_s = np.column_stack([X_train[:, :1], scaler.fit_transform(X_train[:, 1:])])
        X_test_s = np.column_stack([X_test[:, :1], scaler.transform(X_test[:, 1:])])

        model = Ridge(lam=lam, fit_intercept_column=True).fit(X_train_s, y_train)
        weights.append(model.coef_)

        y_pred_train = model.predict(X_train_s)
        y_pred_test = model.predict(X_test_s)

        mse_train.append(mean_squared_error(y_train, y_pred_train))
        mse_test.append(mean_squared_error(y_test, y_pred_test))
        r2_train.append(r2_score(y_train, y_pred_train))
        r2_test.append(r2_score(y_test, y_pred_test))

    return {
        "degrees": degrees_arr,
        "weights": weights,
        "mse_train": np.array(mse_train),
        "mse_test": np.array(mse_test),
        "r2_train": np.array(r2_train),
        "r2_test": np.array(r2_test),
    }

### Baseline experiment: degree × $\lambda$ sweep

We run `fit_polynomial_ridge_by_degree` once per $\lambda$, at the same $n=100$,
$\sigma=0.1$ dataset and degree range (1–15) as part a). `Ridge` solves the
per-observation cost $(1/n)\|y-X\theta\|^2+\lambda\|\theta\|^2$, so $\lambda$ is
compared against $n_{\text{train}}=80$, not 1; the grid spans seven orders of
magnitude, from $0$ (no penalty, i.e. OLS) up to $10^{-1}$ (visibly
over-regularized), log-spaced so the plots below show both the "too little" and
"too much" regularization regimes.

In [ ]:
degrees = range(1, 16)
lambdas = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

# results_by_lambda[lam] is the same dict shape returned by
# fit_polynomial_ols_by_degree in part a): "degrees", "weights", "mse_train",
# "mse_test", "r2_train", "r2_test".
results_by_lambda = {lam: fit_polynomial_ridge_by_degree(x, y, degrees, lam) for lam in lambdas}

## MSE and R2 vs. polynomial degree, by $\lambda$

Same plot as in part a), but now with one curve per $\lambda$. $\lambda = 0$ is the
OLS curve from part a) (Ridge with no penalty).

In [ ]:
degrees_arr = np.array(list(degrees))
# One color per lambda, ordered light -> dark so increasing regularization reads
# as increasing color intensity across both panels.
colors = plt.get_cmap("viridis")(np.linspace(0, 1, len(lambdas)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for lam, color in zip(lambdas, colors, strict=True):
    res = results_by_lambda[lam]
    axes[0].plot(
        res["degrees"],
        res["mse_test"],
        marker="o",
        markersize=3,
        color=color,
        label=rf"$\lambda={lam:g}$",
    )
    axes[1].plot(
        res["degrees"],
        res["r2_test"],
        marker="o",
        markersize=3,
        color=color,
        label=rf"$\lambda={lam:g}$",
    )

# Log scale: test MSE spans several orders of magnitude between low and high
# degree, which would otherwise flatten the low-degree differences.
axes[0].set_yscale("log")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("Test MSE")
axes[0].set_title(r"Test MSE vs. degree, by $\lambda$")
axes[0].legend(fontsize="small")

axes[1].axhline(0, color="gray", lw=0.5)  # R2 = 0: no better than predicting mean(y)
axes[1].set_xlabel("Polynomial degree")
axes[1].set_ylabel(r"Test $R^2$")
axes[1].set_title(r"Test $R^2$ vs. degree, by $\lambda$")
axes[1].legend(fontsize="small")

fig.tight_layout()
plt.show()

## Comparison with OLS (part a))

$\lambda = 0$ should make `Ridge` reduce to `OLS` exactly (both solve the same
normal equations, just via `np.linalg.solve` vs. the SVD pseudoinverse), which we
check numerically below for one degree.

> **TODO:** Compare the `results_by_lambda[0.0]` test-MSE/R2 curves above against
> the actual `ols_experiment.ipynb` (part a)) results degree-by-degree, and discuss
> where a small $\lambda>0$ helps (variance reduction at high degree) and where it
> hurts (bias at low degree, where OLS is already well-determined).

In [ ]:
# Sanity check: lam=0 Ridge should reduce to OLS (up to solver differences).
# Rebuilt independently here (rather than reusing results_by_lambda) so this cell
# is a self-contained check with its own explicit design matrix and split.
degree_check = 10
X_check = univariate_polynomial_design_matrix(x=x, degree=degree_check)
X_train_check, X_test_check, y_train_check, y_test_check = train_test_split(
    X_check, y, test_size=0.2, random_state=42
)

scaler_check = StandardScaler()
X_train_check_s = np.column_stack(
    [X_train_check[:, :1], scaler_check.fit_transform(X_train_check[:, 1:])]
)

theta_ridge0 = Ridge(lam=0.0, fit_intercept_column=True).fit(X_train_check_s, y_train_check).coef_
theta_ols = OLS().fit(X_train_check_s, y_train_check).coef_

# Expect this to be at the level of solver round-off (~1e-9), not exactly 0:
# Ridge solves via np.linalg.solve, OLS via the SVD pseudoinverse.
print("max |theta_ridge(lam=0) - theta_ols| =", np.max(np.abs(theta_ridge0 - theta_ols)))

## Ridge trace: coefficients vs. $\lambda$

At a fixed, deliberately over-parameterized degree (15), fit Ridge across a dense
grid of $\lambda$ and trace each coefficient $\theta_i$ as $\lambda$ grows. This is
the classic "ridge trace" plot: it should show every coefficient (other than the
unpenalized intercept) shrinking monotonically toward zero as $\lambda \to \infty$.

In [ ]:
degree_trace = 15  # highest degree from the sweep: most over-parameterized, most shrinkage
# 1e-8 is effectively unpenalized (theta ~ OLS, ill-conditioned at this degree);
# 1e-2 shrinks essentially every coefficient to ~0 (n_train*lambda ~ 1 vs. s_i^2).
lambdas_trace = np.logspace(-8, -2, 60)

X_trace_full = univariate_polynomial_design_matrix(x=x, degree=degree_trace)
X_train_trace, X_test_trace, y_train_trace, y_test_trace = train_test_split(
    X_trace_full, y, test_size=0.2, random_state=42
)

scaler_trace = StandardScaler()
X_train_trace_s = np.column_stack(
    [X_train_trace[:, :1], scaler_trace.fit_transform(X_train_trace[:, 1:])]
)

# One fit per lambda; theta_trace[k] is the degree-15 coefficient vector at lambdas_trace[k].
theta_trace = np.array(
    [
        Ridge(lam=lam, fit_intercept_column=True).fit(X_train_trace_s, y_train_trace).coef_
        for lam in lambdas_trace
    ]
)

fig, ax = plt.subplots(figsize=(8, 5))
for coef_idx in range(1, degree_trace + 1):  # skip the unpenalized intercept (index 0)
    ax.plot(lambdas_trace, theta_trace[:, coef_idx], lw=1, label=rf"$\theta_{{{coef_idx}}}$")

ax.set_xscale("log")
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel(r"Coefficient value $\theta_i$")
ax.set_title(rf"Ridge trace: coefficients vs. $\lambda$ (degree {degree_trace})")
ax.legend(ncol=2, fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
plt.show()

## Shrinkage of singular-value modes

Write the (standardized, non-intercept) training design matrix via its SVD,
$X = U S V^T$. The OLS solution is

$$
\hat{\theta}_{\mathrm{OLS}} = V S^{-1} U^T y = \sum_i \frac{1}{s_i}\left(u_i^T y\right) v_i,
$$

while the Ridge solution (Section 3.8 of the lecture notes) is

$$
\hat{\theta}_{\mathrm{ridge}} = V\,\mathrm{diag}\!\left(\frac{s_i}{s_i^2+n\lambda}\right) U^T y
 = \sum_i f_i(\lambda) \cdot \frac{1}{s_i}\left(u_i^T y\right) v_i,
\qquad f_i(\lambda) = \frac{s_i^2}{s_i^2+n\lambda} \in [0, 1].
$$

The $n\lambda$ (rather than plain $\lambda$) comes from `Ridge` solving the
per-observation cost $(1/n)\|y - X\theta\|^2 + \lambda\|\theta\|^2$ (Section 3.10),
whose normal equations are $(X^TX + n\lambda I)\theta = X^Ty$.

So Ridge is exactly OLS with every SVD mode $v_i$ shrunk by a factor $f_i(\lambda)$:
modes with a large singular value $s_i$ (well-determined, low-variance directions)
are barely shrunk, while modes with a small $s_i$ (poorly-determined, high-variance,
noise-sensitive directions — typically the highest-order polynomial terms) are
shrunk hardest. This is exactly the mechanism behind the coefficient shrinkage seen
in the ridge trace above, and behind the test-MSE improvement at high polynomial
degree in the earlier plot.

In [ ]:
# Non-intercept, standardized columns of the same (degree 15) training design
# matrix used for the ridge trace above.
X_shrink = X_train_trace_s[:, 1:]
n_shrink = X_shrink.shape[0]
# np.linalg.svd returns singular values sorted in decreasing order already.
singular_values = np.linalg.svd(X_shrink, compute_uv=False)

fig, ax = plt.subplots(figsize=(7, 4))
mode_idx = np.arange(1, len(singular_values) + 1)
# A handful of representative lambdas, extending a bit past the ridge trace's
# range to also show modes fully shrunk to ~0.
for lam in [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    shrinkage = singular_values**2 / (singular_values**2 + n_shrink * lam)
    ax.plot(mode_idx, shrinkage, marker="o", markersize=3, label=rf"$\lambda={lam:g}$")

ax.set_xlabel(r"Singular-value mode index (decreasing $s_i$)")
ax.set_ylabel(r"Shrinkage factor $f_i(\lambda) = s_i^2/(s_i^2+n\lambda)$")
ax.set_title(rf"Shrinkage of SVD modes vs. $\lambda$ (degree {degree_trace})")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

> **TODO:** Part (b) asks for a discussion of the dependence on $\lambda$ tied to
> this shrinkage picture, and a comparison with the OLS results of part a). Points
> to cover once the plots above are finalized:
>
> - How the optimal $\lambda$ (lowest test MSE) shifts with polynomial degree —
>   is more regularization needed as the model gets more over-parameterized?
> - Read off the shrinkage-factor plot: which mode indices are still near
>   $f_i \approx 1$ (unshrunk) vs. $f_i \approx 0$ (killed) at the $\lambda$ values
>   that performed best above.
> - Whether Ridge actually improves on OLS's test MSE at high degree, and whether
>   it costs anything (bias) at low degree where OLS was already well-conditioned.
> - Optionally, repeat the $n$-sweep from part a) (`n_values = [50, 100, 300, 500,
>   1000]`) for Ridge, to see whether the benefit of regularization shrinks as more
>   data makes $X^TX$ better conditioned on its own.